In [2]:
from core.service import RAG
from langchain_core.messages import SystemMessage

from repository import VectorRepositoryChromaDB
from registry import ModelRegistry, wire_llm_system_instruction

model = ModelRegistry().construct_model('gemini-3.5-flash-lite')
document_repository = VectorRepositoryChromaDB()
rag_service = RAG(document_repository, model, wire_llm_system_instruction())

ModuleNotFoundError: No module named 'core'

In [6]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.prebuilt import ToolNode

repository_tools = [document_repository.ranking_search, document_repository.get_search_filters]
model_with_tools = model.bind_tools(repository_tools)
tools_node = ToolNode(repository_tools)
system_instruction = wire_llm_system_instruction()

def call_llm(state: MessagesState):
    system_message = SystemMessage(content=system_instruction)
    response = model_with_tools.invoke([system_message] + state['messages'])
    return {'messages': response}

def check_continue(state: MessagesState):
    last_message = state['messages'][-1]
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        return "tool_call"
    return END

workflow = StateGraph(MessagesState)

workflow.add_node("agent", call_llm)
workflow.add_node("tools", tools_node)
workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", check_continue, {"tool_call": "tools", END: END})
workflow.add_edge("tools", "agent")

environment = workflow.compile(checkpointer=MemorySaver())


In [5]:
initial_input = {
    "messages":[("user", "summarize me china ai policies relating to teenager safety")]
}

config = {"configurable": {"thread_id": "session_1"}}

result = environment.invoke(initial_input, config=config)

print(result["messages"][-1].text)

China’s policies regarding teenager and minor safety in the context of Artificial Intelligence—particularly Generative AI—are structured around preventing addiction, safeguarding physical and mental health, and restricting inappropriate content or commercial transactions. 

The primary policy frameworks and standards governing this area include:

### 1. **Interim Measures for the Management of Generative Artificial Intelligence Services (Enacted)**
* **Anti-Addiction Measures:** Providers are legally required to employ effective measures to guide minor users toward scientific understanding and lawful use of generative AI, explicitly preventing overreliance or addiction.
* **Service Guidance:** Providers must clearly define and disclose user groups, occasions, and uses for their services.

### 2. **National Standard: Basic Safety Requirements for Generative AI Services (Draft for Feedback)**
This national standard builds upon the Interim Measures by establishing more granular, technical